In [1]:
from docx import Document
from transformers import AutoTokenizer

# === Шаг 1. Извлечь жирные сущности и текст ===
def extract_bold_entities(docx_path):
    doc = Document(docx_path)
    sentences = []
    bold_entities = set()

    for para in doc.paragraphs:
        sentence = ""
        for run in para.runs:
            text = run.text
            if run.bold and text.strip():
                bold_entities.add(text.strip())
            sentence += text
        if sentence.strip():
            sentences.append(sentence.strip())
    return sentences, bold_entities

# === Шаг 2. Загрузить файл с сущностями и категориями ===
def load_entity_labels(file_path):
    entity_to_label = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().rsplit(' ', 1)
            if len(parts) == 2:
                entity, label = parts
                entity_to_label[entity.strip()] = label.strip()
    return entity_to_label

# === Шаг 3. Связать жирные сущности с категориями ===
def match_entities_with_labels(bold_entities, entity_to_label):
    matched = {}
    unmatched = []

    for entity in bold_entities:
        if entity in entity_to_label:
            matched[entity] = entity_to_label[entity]
        else:
            unmatched.append(entity)
    return matched, unmatched

# === Шаг 4. Основной код ===
if __name__ == "__main__":
    docx_path = "texts/entities.docx"
    labels_path = "texts/entities_metka.txt"
    # Загружаем
    sentences, bold_entities = extract_bold_entities(docx_path)
    entity_to_label = load_entity_labels(labels_path)
    matched, unmatched = match_entities_with_labels(bold_entities, entity_to_label)

    # Показываем результат
    print("\n📌 Найденные сущности с категориями:")
    for entity, label in matched.items():
        print(f"  {entity} → {label}")

    if unmatched:
        print("\n Сущности, для которых НЕ найдены категории:")
        for entity in unmatched:
            print(f"  {entity}")

    print("\n📄 Всего предложений:", len(sentences))
    print("Пример предложений:")
    for i, s in enumerate(sentences[:5]):
        print(f"  {i+1}: {s}")


/home/nika/Documents/Code/NER/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📌 Найденные сущности с категориями:
  Билибиным → PER
  Мюрата → PER
  de Berlin → GPE
  венский мост → FAC
  Мортье → PER
  Владимира с бантом → VEH
  Брюнн → GPE
  венско-цнаймскую → FAC
  ваш Петр → PER
  Кремса → GPE
  Суворове → PER
  деревне Зальценек → GPE
  Долгоруков → PER
  Андрей Болконский → PER
  Павлоградского гусарского полков → ORG
  der Donau → GPE
  эрцгерцог Фердинанд → PER
  Дунай → LOC
  Вена → GPE
  Александра Первого → PER
  Болконский → PER
  Несвицкого → PER
  Грунт → GPE
  Иван Лукич → PER
  pont de Vienne → FAC
  Браунау → GPE
  Цнайм → GPE
  императору Францу → PER
  Матвевны → VEH
  Государь → PER
  Антонов → PER
  d’Ulm → GPE
  Бандарчука → PER
  Гольдбахом → LOC
  Вене → GPE
  Андрею → PER
  pont de Thabor → FAC
  Экономовым → PER
  Г’остов → PER
  Репнина → PER
  фрейлейн Матильду → PER
  императора Франца → PER
  Мюрат → PER
  de Vienne → GPE
  Мюрату → PER
  князем Ипполитом Курагиным → PER
  Остралиц → GPE
  Дунае → LOC
  Le prince d’Auersperg → PER


In [2]:
import os
import random
import re
from pathlib import Path
from docx import Document
from transformers import AutoTokenizer

# === Пути ===
docx_path = "texts/entities.docx"
labels_path = "texts/entities_metka.txt"
output_dir = Path("rubert.ipynb/datasets")
output_dir.mkdir(parents=True, exist_ok=True)

# === Шаг 1: Извлечение жирных сущностей и текста ===
def extract_bold_entities(docx_path):
    doc = Document(docx_path)
    sentences = []
    bold_entities = set()

    for para in doc.paragraphs:
        sentence = ""
        for run in para.runs:
            text = run.text
            if run.bold and text.strip():
                bold_entities.add(text.strip())
            sentence += text

        # Разбиваем абзац на предложения
        if sentence.strip():
            split_sentences = re.split(r'(?<=[.!?])\s+', sentence.strip())
            sentences.extend(split_sentences)

    return sentences, bold_entities

# === Шаг 2: Загрузка категорий сущностей ===
def load_entity_labels(file_path):
    entity_to_label = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().rsplit(' ', 1)
            if len(parts) == 2:
                entity, label = parts
                entity_to_label[entity.strip()] = label.strip()
    return entity_to_label

# === Шаг 3: BIO-разметка с учетом offset-ов ===
def tokenize_and_tag_sentences(sentences, matched_entities, tokenizer):
    data = []
    for sentence in sentences:
        encoding = tokenizer(
            sentence,
            return_offsets_mapping=True,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors=None,
        )

        input_ids = encoding["input_ids"]
        offsets = encoding["offset_mapping"]
        tokens = tokenizer.convert_ids_to_tokens(input_ids)
        labels = ["O"] * len(tokens)

        for entity, tag in matched_entities.items():
            if not entity.strip():
                continue
            start_idx = 0
            while True:
                start = sentence.find(entity, start_idx)
                if start == -1:
                    break
                end = start + len(entity)

                for i, (token_start, token_end) in enumerate(offsets):
                    if token_start is None or token_end is None:
                        continue
                    if token_start >= start and token_end <= end:
                        if labels[i] == "O":
                            labels[i] = f"B-{tag}" if token_start == start else f"I-{tag}"
                start_idx = end

        data.append((tokens, labels))
    return data

# === Поддержка: объединение субтокенов ===
def merge_subword_tokens(tokens, labels):
    merged_tokens = []
    merged_labels = []

    for token, label in zip(tokens, labels):
        if token.startswith("##") and merged_tokens:
            merged_tokens[-1] += token[2:]
        else:
            merged_tokens.append(token)
            merged_labels.append(label)
    return merged_tokens, merged_labels

# === Шаг 4: Сохранение в .txt ===
def save_to_txt(data, path, tokenizer):
    with open(path, "w", encoding="utf-8") as f:
        for tokens, labels in data:
            tokens, labels = merge_subword_tokens(tokens, labels)
            for t, l in zip(tokens, labels):
                if t in tokenizer.all_special_tokens:
                    continue
                f.write(f"{t} {l}\n")
            f.write("\n")  # Пустая строка между предложениями

# === Шаг 5: Основной запуск ===
if __name__ == "__main__":
    # Загрузка текста и сущностей
    sentences, bold_entities = extract_bold_entities(docx_path)
    entity_to_label = load_entity_labels(labels_path)

    matched = {e: entity_to_label[e] for e in bold_entities if e in entity_to_label}
    tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

    print(f"📄 Найдено предложений: {len(sentences)}")
    print(f"🏷  Жирных сущностей с категорией: {len(matched)}")

    # BIO-разметка
    data = tokenize_and_tag_sentences(sentences, matched, tokenizer)

    # Перемешиваем и делим
    random.seed(42)
    random.shuffle(data)
    n = len(data)
    train_data = data[:int(n * 0.7)]
    val_data = data[int(n * 0.7):int(n * 0.85)]
    test_data = data[int(n * 0.85):]

    # Сохраняем
    save_to_txt(train_data, output_dir / "train.txt", tokenizer)
    save_to_txt(val_data, output_dir / "val.txt", tokenizer)
    save_to_txt(test_data, output_dir / "test.txt", tokenizer)

    print("\n✅ Готово! BIO-разметка сохранена в:")
    print(f" - {output_dir / 'train.txt'}")
    print(f" - {output_dir / 'val.txt'}")
    print(f" - {output_dir / 'test.txt'}")


📄 Найдено предложений: 3114
🏷  Жирных сущностей с категорией: 444

✅ Готово! BIO-разметка сохранена в:
 - rubert.ipynb/datasets/train.txt
 - rubert.ipynb/datasets/val.txt
 - rubert.ipynb/datasets/test.txt


In [3]:
import os
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import DatasetDict, Dataset
from sklearn.metrics import classification_report
from seqeval.metrics import f1_score, classification_report as seqeval_classification_report

# === Пути ===
data_dir = Path("rubert.ipynb/datasets")
model_checkpoint = "DeepPavlov/rubert-base-cased"
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-GPE', 'I-GPE', 'B-FAC', 'I-FAC', 'B-VEH', 'I-VEH']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# === Загрузка BIO-данных из файлов ===
def load_bio_dataset(data_dir):
    def read_file(file_path):
        tokens = []
        labels = []
        with open(file_path, encoding='utf-8') as f:
            tok_seq = []
            lab_seq = []
            for line in f:
                line = line.strip()
                if not line:
                    if tok_seq:
                        tokens.append(tok_seq)
                        labels.append(lab_seq)
                        tok_seq, lab_seq = [], []
                else:
                    token, label = line.split()
                    tok_seq.append(token)
                    lab_seq.append(label)
        return {"tokens": tokens, "ner_tags": labels}

    return DatasetDict({
        "train": Dataset.from_dict(read_file(data_dir / "train.txt")),
        "validation": Dataset.from_dict(read_file(data_dir / "val.txt")),
        "test": Dataset.from_dict(read_file(data_dir / "test.txt")),
    })

# === Токенизатор ===
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# === Преобразуем строки меток в ID ===
def encode_labels(example):
    tokens = example["tokens"]
    labels = example["ner_tags"]
    tokenized = tokenizer(tokens, is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[labels[word_idx]])
        else:
            # В случае субтокена
            label = labels[word_idx]
            if label.startswith("B-"):
                label = label.replace("B-", "I-")
            aligned_labels.append(label2id[label])
        previous_word_idx = word_idx
    tokenized["labels"] = aligned_labels
    return tokenized

# === Загружаем данные ===
dataset = load_bio_dataset(data_dir)
encoded_dataset = dataset.map(encode_labels, batched=False)

# === Модель ===
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# === Аргументы тренировки ===
training_args = TrainingArguments(
    output_dir="./rubert-ner",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

# === Вычисление метрик ===
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        cur_labels = []
        cur_preds = []
        for p, l in zip(pred, lab):
            if l != -100:
                cur_labels.append(id2label[l])
                cur_preds.append(id2label[p])
        true_labels.append(cur_labels)
        true_predictions.append(cur_preds)

    f1 = f1_score(true_labels, true_predictions)
    print(seqeval_classification_report(true_labels, true_predictions))
    return {"f1": f1}

# === Обучение ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./rubert-ner/final")
tokenizer.save_pretrained("./rubert-ner/final")


2025-04-14 16:27:59.208447: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-14 16:27:59.215427: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-14 16:27:59.264938: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-14 16:27:59.312671: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744637279.353197    7305 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744637279.36

Epoch,Training Loss,Validation Loss,F1
1,0.043800,0.031554,0.893417
2,0.030700,0.025231,0.913386
3,0.013000,0.022103,0.919255
4,0.008900,0.020790,0.929577


              precision    recall  f1-score   support

         FAC       0.00      0.00      0.00         1
         GPE       0.73      0.85      0.79        26
         LOC       0.60      0.60      0.60         5
         ORG       0.00      0.00      0.00         1
         PER       0.90      0.94      0.92       277
         VEH       0.00      0.00      0.00         4

   micro avg       0.88      0.91      0.89       314
   macro avg       0.37      0.40      0.38       314
weighted avg       0.86      0.91      0.89       314



/home/nika/Documents/Code/NER/.venv/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

         FAC       1.00      1.00      1.00         1
         GPE       0.76      0.85      0.80        26
         LOC       1.00      0.80      0.89         5
         ORG       0.00      0.00      0.00         1
         PER       0.92      0.95      0.93       277
         VEH       0.50      0.25      0.33         4

   micro avg       0.90      0.92      0.91       314
   macro avg       0.70      0.64      0.66       314
weighted avg       0.90      0.92      0.91       314

              precision    recall  f1-score   support

         FAC       1.00      1.00      1.00         1
         GPE       0.82      0.88      0.85        26
         LOC       1.00      0.80      0.89         5
         ORG       1.00      1.00      1.00         1
         PER       0.91      0.96      0.93       277
         VEH       0.33      0.25      0.29         4

   micro avg       0.90      0.94      0.92       314
   macro avg       0.84

('./rubert-ner/final/tokenizer_config.json',
 './rubert-ner/final/special_tokens_map.json',
 './rubert-ner/final/vocab.txt',
 './rubert-ner/final/added_tokens.json',
 './rubert-ner/final/tokenizer.json')

In [4]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Загрузка токенизатора и модели
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
model = AutoModelForTokenClassification.from_pretrained("./rubert-ner/final")  # Укажите путь к вашей модели

# Новый текст для предсказания
new_sentences = [
    "Тимохин и Наташа Ростова молчал, и лицо его так было неприятно, что Пьер обращался более к добродушному батальонному командиру Тимохину, чем к Болконскому.",
    "29-го мая Наполеон выехал из Дрездена, где он пробыл три недели, окруженный двором, составленным из принцев, герцогов, королей и даже одного императора.",
    "Кутузов сидел, понурив седую голову и опустившись тяжелым телом, на покрытой ковром лавке, на том самом месте, на котором утром его видел Пьер. Он не делал никаких распоряжении, а только соглашался или не соглашался на то, что предлагали ему."
]

# Токенизация предложений
inputs = tokenizer(new_sentences, padding=True, truncation=True, return_tensors="pt")

# Получение предсказаний (выход модели)
with torch.no_grad():  # Отключаем вычисление градиентов
    outputs = model(**inputs)
    predictions = outputs.logits

# Получаем индексы наиболее вероятных меток
predicted_labels = torch.argmax(predictions, dim=-1)

# Преобразуем индексы в метки
label_map = model.config.id2label  # Сопоставление индекса метки и её имени
predicted_labels = predicted_labels.cpu().numpy()

# Печатаем результаты
for i, (sentence, label_seq) in enumerate(zip(new_sentences, predicted_labels)):
    # Преобразуем id в токены для текущего предложения
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][i])
    
    print(f"Предсказания для: {sentence}")
    for token, label in zip(tokens, label_seq):
        if token.startswith("##"):  # Пропускаем субтокены
            continue
        print(f"{token} => {label_map[label]}")
    print()  # Пустая строка между предложениями


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Предсказания для: Тимохин и Наташа Ростова молчал, и лицо его так было неприятно, что Пьер обращался более к добродушному батальонному командиру Тимохину, чем к Болконскому.
[CLS] => I-PER
Тимо => B-PER
и => O
Наташа => B-PER
Ростова => I-PER
молчал => O
, => O
и => O
лицо => O
его => O
так => O
было => O
неприят => O
, => O
что => O
Пьер => B-PER
обращался => O
более => O
к => O
добродуш => O
батальон => O
командиру => O
Тимо => B-PER
, => O
чем => O
к => O
Бол => B-PER
. => O
[SEP] => I-PER
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => B-PER
[PAD] => B-PER
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => B-PER
[PAD] => I-PER
[PAD] => B-PER
[PAD] => O
[PAD] => B-PER
[PAD] => O

Предсказания для: 29-го мая Наполеон выехал из Дрездена, где он пробыл три недели, окруженный двором, составленным из принцев, герцогов, королей и даже одного императора.
[CLS] => O
29 => O
- => O
го => O
мая => O
Наполеон => B-PER
выехал => O
из => O
Дрездена => B-GPE
, => O
где => O
он =

In [2]:
import os
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import DatasetDict, Dataset
from sklearn.metrics import classification_report, confusion_matrix
from seqeval.metrics import f1_score, classification_report as seqeval_classification_report
import pandas as pd

# === Пути ===
data_dir = Path("rubert.ipynb/datasets")
model_checkpoint = "DeepPavlov/rubert-base-cased"
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-GPE', 'I-GPE', 'B-FAC', 'I-FAC', 'B-VEH', 'I-VEH']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# === Загрузка BIO-данных из файлов ===
def load_bio_dataset(data_dir):
    def read_file(file_path):
        tokens = []
        labels = []
        with open(file_path, encoding='utf-8') as f:
            tok_seq = []
            lab_seq = []
            for line in f:
                line = line.strip()
                if not line:
                    if tok_seq:
                        tokens.append(tok_seq)
                        labels.append(lab_seq)
                        tok_seq, lab_seq = [], []
                else:
                    token, label = line.split()
                    tok_seq.append(token)
                    lab_seq.append(label)
        return {"tokens": tokens, "ner_tags": labels}

    return DatasetDict({
        "train": Dataset.from_dict(read_file(data_dir / "train.txt")),
        "validation": Dataset.from_dict(read_file(data_dir / "val.txt")),
        "test": Dataset.from_dict(read_file(data_dir / "test.txt")),
    })

# === Токенизатор ===
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# === Преобразуем строки меток в ID ===
def encode_labels(example):
    tokens = example["tokens"]
    labels = example["ner_tags"]
    tokenized = tokenizer(tokens, is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[labels[word_idx]])
        else:
            # В случае субтокена заменяем B- на I-
            label = labels[word_idx]
            if label.startswith("B-"):
                label = label.replace("B-", "I-")
            aligned_labels.append(label2id[label])
        previous_word_idx = word_idx
    tokenized["labels"] = aligned_labels
    return tokenized

# === Загружаем данные ===
dataset = load_bio_dataset(data_dir)
encoded_dataset = dataset.map(encode_labels, batched=False)

# === Модель ===
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# === Аргументы тренировки ===
training_args = TrainingArguments(
    output_dir="./rubert-ner",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

# === Вычисление метрик (с добавлением конфузионной матрицы) ===
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    # Собираем список истинных меток и предсказаний на уровне сущностей (для seqeval)
    for pred, lab in zip(predictions, labels):
        cur_labels = []
        cur_preds = []
        for p, l in zip(pred, lab):
            if l != -100:
                cur_labels.append(id2label[l])
                cur_preds.append(id2label[p])
        true_labels.append(cur_labels)
        true_predictions.append(cur_preds)

    # Вычисляем f1-меру с помощью seqeval
    f1 = f1_score(true_labels, true_predictions)
    print("Seqeval Classification Report:")
    print(seqeval_classification_report(true_labels, true_predictions))
    
    # Для конфузионной матрицы вычисляем метрики на уровне токенов:
    flat_true = [label for seq in true_labels for label in seq]
    flat_pred = [pred for seq in true_predictions for pred in seq]
    cm = confusion_matrix(flat_true, flat_pred, labels=label_list)
    cm_df = pd.DataFrame(cm, index=label_list, columns=label_list)
    print("Confusion Matrix:")
    print(cm_df)
    
    return {"f1": f1}

# === Обучение ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./rubert-ner/final")
tokenizer.save_pretrained("./rubert-ner/final")


Map: 100%|██████████| 468/468 [00:00<00:00, 6920.42 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/nika/Documents/Code/NER/.venv/lib/python3.10/site-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_9877/1559627396.py:135: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,0.044500,0.032042,0.875981
2,0.032100,0.028977,0.931638
3,0.016100,0.022984,0.936709
4,0.009300,0.022176,0.939683


Seqeval Classification Report:
              precision    recall  f1-score   support

         FAC       0.00      0.00      0.00         1
         GPE       0.69      0.85      0.76        26
         LOC       0.25      0.20      0.22         5
         ORG       0.00      0.00      0.00         1
         PER       0.89      0.92      0.91       277
         VEH       0.00      0.00      0.00         4

   micro avg       0.86      0.89      0.88       314
   macro avg       0.30      0.33      0.31       314
weighted avg       0.85      0.89      0.87       314

Confusion Matrix:
          O  B-PER  I-PER  B-ORG  I-ORG  B-LOC  I-LOC  B-GPE  I-GPE  B-FAC  \
O      9344      6      4      0      0      0      0      1      4      0   
B-PER     6    265      6      0      0      0      0      0      0      0   
I-PER    13      7    209      0      0      0      0      0      2      0   
B-ORG     1      0      0      0      0      0      0      0      0      0   
I-ORG     3      0

/home/nika/Documents/Code/NER/.venv/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Seqeval Classification Report:
              precision    recall  f1-score   support

         FAC       1.00      1.00      1.00         1
         GPE       0.92      0.85      0.88        26
         LOC       1.00      0.80      0.89         5
         ORG       1.00      1.00      1.00         1
         PER       0.93      0.96      0.94       277
         VEH       0.00      0.00      0.00         4

   micro avg       0.93      0.93      0.93       314
   macro avg       0.81      0.77      0.79       314
weighted avg       0.92      0.93      0.93       314

Confusion Matrix:
          O  B-PER  I-PER  B-ORG  I-ORG  B-LOC  I-LOC  B-GPE  I-GPE  B-FAC  \
O      9349      1      3      0      0      0      0      1      5      0   
B-PER     5    269      3      0      0      0      0      0      0      0   
I-PER     9      5    217      0      0      0      0      0      0      0   
B-ORG     0      0      0      0      1      0      0      0      0      0   
I-ORG     0      0

('./rubert-ner/final/tokenizer_config.json',
 './rubert-ner/final/special_tokens_map.json',
 './rubert-ner/final/vocab.txt',
 './rubert-ner/final/added_tokens.json',
 './rubert-ner/final/tokenizer.json')

In [7]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
extract_entities.py
~~~~~~~~~~~~~~~~~~~
Извлекает и подсчитывает все сущности из .docx-файла
и сохраняет их в .txt (формат: <текст>\t<тип>\t<частота>).
"""

import torch
from pathlib import Path
from collections import Counter
from itertools import chain
from docx import Document
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ---------- Параметры ----------
MODEL_DIR   = Path("rubert-ner/final")                 # каталог сохранённой модели
DOCX_PATH   = Path("texts/Бородино.Чистый текст.docx") # исходный .docx
OUTPUT_PATH = Path("texts/Бородино_entities.txt")      # файл с результатом
MAX_TOKENS  = 490                                      # длина чанка (<512)

# ---------- Загрузка модели ----------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
model.eval()

device_id = 0 if torch.cuda.is_available() else -1
ner_pipe  = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",   # объединяем B/I-теги
    device=device_id
)

# ---------- Чтение документа ----------
doc = Document(str(DOCX_PATH))
full_text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())

# ---------- Функция чанкирования ----------
def chunk_text(text: str, max_len: int = MAX_TOKENS):
    """Разбивает текст на чанки ≤max_len токенов."""
    words, cur, chunks, cur_len = text.split(), [], [], 0
    for w in words:
        w_len = len(tokenizer.tokenize(w))
        if cur_len + w_len > max_len:
            chunks.append(" ".join(cur))
            cur, cur_len = [w], w_len
        else:
            cur.append(w)
            cur_len += w_len
    if cur:
        chunks.append(" ".join(cur))
    return chunks

chunks = chunk_text(full_text)
print(f"Чанков к обработке: {len(chunks)}")

# ---------- Извлечение сущностей ----------
batch_size = 16
entities_raw = []
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    batch_out = ner_pipe(batch)          # List[List[Dict]]
    entities_raw.extend(chain.from_iterable(batch_out))

print(f"✓ Сущностей извлечено: {len(entities_raw)}")

# ---------- Подсчёт частот ----------
counter = Counter((ent["word"], ent["entity_group"]) for ent in entities_raw)

# ---------- Запись результата ----------
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for (text, ent_type), freq in counter.most_common():
        f.write(f"{text}\t{ent_type}\t{freq}\n")

print(f"Готово! Найдено {len(counter)} уникальных сущностей. "
      f"Результат сохранён в «{OUTPUT_PATH}».")


Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Чанков к обработке: 76
✓ Сущностей извлечено: 1144
Готово! Найдено 335 уникальных сущностей. Результат сохранён в «texts/Бородино_entities.txt».
